In [1]:
from pathlib import Path

import mltrainer
import torch
from mltrainer import Trainer, rnn_models
from torch import optim

mltrainer.__version__

'0.2.5'

# 1 Iterators
We will be using an interesting dataset. [link](https://tev.fbk.eu/resources/smartwatch)

From the site:
> The SmartWatch Gestures Dataset has been collected to evaluate several gesture recognition algorithms for interacting with mobile applications using arm gestures. Eight different users performed twenty repetitions of twenty different gestures, for a total of 3200 sequences. Each sequence contains acceleration data from the 3-axis accelerometer of a first generation Sony SmartWatch™, as well as timestamps from the different clock sources available on an Android device. The smartwatch was worn on the user's right wrist. 


In [2]:
from mads_datasets import DatasetFactoryProvider, DatasetType
from mltrainer.preprocessors import PaddedPreprocessor

preprocessor = PaddedPreprocessor()

gesturesdatasetfactory = DatasetFactoryProvider.create_factory(DatasetType.GESTURES)
streamers = gesturesdatasetfactory.create_datastreamer(batchsize=32, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]

2026-06-16 17:35:49.628 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /Users/vince/.cache/mads_datasets/gestures


  0%|          | 0/2600 [00:00<?, ?it/s]

 58%|█████▊    | 1514/2600 [00:00<00:00, 15129.12it/s]

100%|██████████| 2600/2600 [00:00<00:00, 15220.45it/s]

  0%|          | 0/651 [00:00<?, ?it/s]

100%|██████████| 651/651 [00:00<00:00, 13523.11it/s]

In [3]:
len(train), len(valid)

(81, 20)

In [4]:
trainstreamer = train.stream()
validstreamer = valid.stream()
x, y = next(iter(trainstreamer))
x.shape, y

(torch.Size([32, 32, 3]),
 tensor([ 6,  4,  5, 17, 19, 15,  1, 16,  2,  7,  1,  0,  1, 11,  7, 17,  3,  8,
          3, 18, 12,  7, 14,  8,  8,  5,  4,  7,  7,  2, 14,  0]))

Can you make sense of the shape?
What does it mean that the shapes are sometimes (32, 27, 3), but a second time might look like (32, 30, 3)? In other words, the second (or first, if you insist on starting at 0) dimension changes. Why is that? How does the model handle this? Do you think this is already padded, or still has to be padded?


# 2 Excercises
Lets test a basemodel, and try to improve upon that.

Fill the gestures.gin file with relevant settings for `input_size`, `hidden_size`, `num_layers` and `horizon` (which, in our case, will be the number of classes...)

As a rule of thumbs: start lower than you expect to need!

In [5]:
from mltrainer import ReportTypes, TrainerSettings
from mltrainer.metrics import Accuracy

accuracy = Accuracy()


In [6]:
model = rnn_models.BaseRNN(
    input_size=3,
    hidden_size=64,
    num_layers=1,
    horizon=20,
)

Test the model. What is the output shape you need? Remember, we are doing classification!

In [7]:
yhat = model(x)
yhat.shape

torch.Size([32, 20])

Test the accuracy

In [8]:
accuracy(y, yhat)

0.09375

What do you think of the accuracy? What would you expect from blind guessing?

Check shape of `y` and `yhat`

In [9]:
yhat.shape, y.shape

(torch.Size([32, 20]), torch.Size([32]))

And look at the output of yhat

In [10]:
yhat[0]

tensor([-0.0071,  0.1810,  0.0773, -0.0287,  0.0432,  0.0890, -0.0759,  0.0973,
         0.0221,  0.0315, -0.0852,  0.0477, -0.0393,  0.0020, -0.0236,  0.1834,
         0.0552, -0.0444, -0.0555, -0.0174], grad_fn=<SelectBackward0>)

Does this make sense to you? If you are unclear, go back to the classification problem with the MNIST, where we had 10 classes.

We have a classification problem, so we need Cross Entropy Loss.
Remember, [this has a softmax built in](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) 

In [11]:
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(yhat, y)
loss

tensor(2.9673, grad_fn=<NllLossBackward0>)

In [12]:
import torch

if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
    print("Using MPS")
elif torch.cuda.is_available():
    device = "cuda:0"
    print("using cuda")
else:
    device = "cpu"
    print("using cpu")

# on my mac, at least for the BaseRNN model, mps does not speed up training
# probably because the overhead of copying the data to the GPU is too high
# so i override the device to cpu
device = "cpu"
# however, it might speed up training for larger models, with more parameters

Using MPS


Set up the settings for the trainer and the different types of logging you want

In [13]:
settings = TrainerSettings(
    epochs=50,
    metrics=[accuracy],
    logdir=Path("gestures"),
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.TOML, ReportTypes.TENSORBOARD, ReportTypes.MLFLOW],
    scheduler_kwargs={"factor": 0.5, "patience": 5},
    earlystop_kwargs = {
        "save": False,
        "verbose": True,
        "patience": 10,
        "delta": 0.0,
    }
)
settings

epochs: 50
metrics: [Accuracy]
logdir: gestures
train_steps: 81
valid_steps: 20
reporttypes: [<ReportTypes.TOML: 'TOML'>, <ReportTypes.TENSORBOARD: 'TENSORBOARD'>, <ReportTypes.MLFLOW: 'MLFLOW'>]
optimizer_kwargs: {'lr': 0.001, 'weight_decay': 1e-05}
scheduler_kwargs: {'factor': 0.5, 'patience': 5}
earlystop_kwargs: {'save': False, 'verbose': True, 'patience': 10, 'delta': 0.0}

In [14]:
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch import Tensor


@dataclass
class ModelConfig:
    input_size: int
    hidden_size: int
    num_layers: int
    output_size: int
    dropout: float = 0.0

class GRUmodel(nn.Module):
    def __init__(
        self,
        config,
    ) -> None:
        super().__init__()
        self.config = config
        self.rnn = nn.GRU(
            input_size=config.input_size,
            hidden_size=config.hidden_size,
            dropout=config.dropout,
            batch_first=True,
            num_layers=config.num_layers,
        )
        self.linear = nn.Linear(config.hidden_size, config.output_size)

    def forward(self, x: Tensor) -> Tensor:
        x, _ = self.rnn(x)
        last_step = x[:, -1, :]
        yhat = self.linear(last_step)
        return yhat

In [15]:
config = ModelConfig(
    input_size=3,
    hidden_size=64,
    num_layers=1,
    output_size=20,
    dropout=0.0,
)


In [16]:
from datetime import datetime

import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("gestures")
modeldir = Path("gestures").resolve()
if not modeldir.exists():
    modeldir.mkdir(parents=True)

with mlflow.start_run():
    mlflow.set_tag("model", "GRU-baseline")
    mlflow.set_tag("dev", "vincent")
    config = ModelConfig(
        input_size=3,
        hidden_size=64,
        num_layers=1,
        output_size=20,
        dropout=0.0,
    )
    mlflow.log_params(config.__dict__)

    model = GRUmodel(
        config=config,
    )

    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=loss_fn,
        optimizer=optim.Adam,
        traindataloader=trainstreamer,
        validdataloader=validstreamer,
        scheduler=optim.lr_scheduler.ReduceLROnPlateau,
        device=device,
    )
    trainer.loop()

    if not settings.earlystop_kwargs["save"]:
        tag = datetime.now().strftime("%Y%m%d-%H%M-")
        modelpath = modeldir / (tag + "model.pt")
        torch.save(model, modelpath)

2026-06-16 17:35:50.371 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures/20260616-173550


2026-06-16 17:35:50.655 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/81 [00:00<?, ?it/s]

 59%|█████▉    | 48/81 [00:00<00:00, 472.12it/s]

...

100%|██████████| 81/81 [00:00<00:00, 455.85it/s]


2026-06-16 17:36:00.156 | INFO     | mltrainer.trainer:report:209 - Epoch 49 train 0.0244 test 0.0741 metric ['0.9812']


2026-06-16 17:36:00.157 | INFO     | mltrainer.trainer:__call__:252 - best loss: 0.0735, current loss 0.0741.Counter 6/10.


100%|██████████| 50/50 [00:09<00:00,  5.17it/s]

100%|██████████| 50/50 [00:09<00:00,  5.27it/s]

Try to update the code above by changing the hyperparameters.
    
To discern between the changes, also modify the tag mlflow.set_tag("model", "new-tag-here") where you add
a new tag of your choice. This way you can keep the models apart.

## LSTM Model

Same structure as GRUmodel, but using `nn.LSTM`. Note that LSTM returns `(output, (h_n, c_n))` instead of `(output, h_n)`.

In [17]:
class LSTMmodel(nn.Module):
    def __init__(
        self,
        config: ModelConfig,
    ) -> None:
        super().__init__()
        self.config = config
        self.rnn = nn.LSTM(
            input_size=config.input_size,
            hidden_size=config.hidden_size,
            dropout=config.dropout,
            batch_first=True,
            num_layers=config.num_layers,
        )
        self.linear = nn.Linear(config.hidden_size, config.output_size)

    def forward(self, x: Tensor) -> Tensor:
        # LSTM returns (output, (h_n, c_n)) unlike GRU which returns (output, h_n)
        x, (_, _) = self.rnn(x)
        last_step = x[:, -1, :]
        yhat = self.linear(last_step)
        return yhat

## Conv1D + GRU Hybrid Model

Conv1D expects input shape `(batch, channels, timesteps)`, but our data arrives as `(batch, timesteps, features=3)`.
We permute before Conv1D, then permute back before feeding into the GRU.

In [18]:
@dataclass
class Conv1DGRUConfig:
    input_size: int
    hidden_size: int
    num_layers: int
    output_size: int
    conv_filters: int = 32
    kernel_size: int = 3
    dropout: float = 0.0


class Conv1DGRUmodel(nn.Module):
    def __init__(
        self,
        config: Conv1DGRUConfig,
    ) -> None:
        super().__init__()
        self.config = config
        # Conv1D: (batch, channels=input_size, timesteps)
        self.conv1 = nn.Conv1d(
            in_channels=config.input_size,
            out_channels=config.conv_filters,
            kernel_size=config.kernel_size,
            padding=config.kernel_size // 2,  # keep timestep dimension
        )
        self.relu = nn.ReLU()
        # GRU takes (batch, timesteps, features=conv_filters)
        self.rnn = nn.GRU(
            input_size=config.conv_filters,
            hidden_size=config.hidden_size,
            dropout=config.dropout,
            batch_first=True,
            num_layers=config.num_layers,
        )
        self.linear = nn.Linear(config.hidden_size, config.output_size)

    def forward(self, x: Tensor) -> Tensor:
        # x shape: (batch, timesteps, features=3)
        x = x.permute(0, 2, 1)  # -> (batch, features=3, timesteps)
        x = self.relu(self.conv1(x))  # -> (batch, conv_filters, timesteps)
        x = x.permute(0, 2, 1)  # -> (batch, timesteps, conv_filters)
        x, _ = self.rnn(x)
        last_step = x[:, -1, :]
        yhat = self.linear(last_step)
        return yhat

## Experiment runs

Update settings to train for more epochs (100) with early stopping and then run multiple experiments comparing GRU, LSTM, and Conv1D+GRU with different hyperparameters.

In [19]:
# Update settings for proper training
settings = TrainerSettings(
    epochs=100,
    metrics=[accuracy],
    logdir=Path("gestures"),
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.TOML, ReportTypes.TENSORBOARD, ReportTypes.MLFLOW],
    scheduler_kwargs={"factor": 0.5, "patience": 5},
    earlystop_kwargs={
        "save": False,
        "verbose": True,
        "patience": 10,
        "delta": 0.0,
    },
)

In [20]:
def run_experiment(
    model: nn.Module,
    model_tag: str,
    config_params: dict,
) -> None:
    """Train a model and log to MLflow."""
    with mlflow.start_run():
        mlflow.set_tag("model", model_tag)
        mlflow.set_tag("dev", "vincent")
        mlflow.log_params(config_params)

        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optim.Adam,
            traindataloader=trainstreamer,
            validdataloader=validstreamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau,
            device=device,
        )
        trainer.loop()

        if not settings.earlystop_kwargs["save"]:
            tag = datetime.now().strftime("%Y%m%d-%H%M-")
            modelpath = modeldir / (tag + "model.pt")
            torch.save(model, modelpath)

### Experiment 1: GRU with larger hidden size and 2 layers

In [21]:
config_gru_large = ModelConfig(
    input_size=3,
    hidden_size=128,
    num_layers=2,
    output_size=20,
    dropout=0.3,
)
model_gru_large = GRUmodel(config=config_gru_large)
run_experiment(model_gru_large, "GRU-h128-l2-d03", config_gru_large.__dict__)

2026-06-16 17:36:00.183 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures/20260616-173600


2026-06-16 17:36:00.183 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/81 [00:00<?, ?it/s]

 15%|█▍        | 12/81 [00:00<00:00, 118.12it/s]

...

2026-06-16 17:36:32.357 | INFO     | mltrainer.trainer:report:209 - Epoch 40 train 0.0026 test 0.0226 metric ['0.9922']


2026-06-16 17:36:32.358 | INFO     | mltrainer.trainer:__call__:252 - best loss: 0.0198, current loss 0.0226.Counter 10/10.


2026-06-16 17:36:32.358 | INFO     | mltrainer.trainer:loop:103 - Interrupting loop due to early stopping patience.


2026-06-16 17:36:32.358 | INFO     | mltrainer.trainer:loop:108 - early_stopping_save was false, using latest model.Set to true to retrieve best model.


 40%|████      | 40/100 [00:32<00:48,  1.24it/s]

### Experiment 2: LSTM baseline

In [22]:
config_lstm = ModelConfig(
    input_size=3,
    hidden_size=128,
    num_layers=2,
    output_size=20,
    dropout=0.3,
)
model_lstm = LSTMmodel(config=config_lstm)
run_experiment(model_lstm, "LSTM-h128-l2-d03", config_lstm.__dict__)

2026-06-16 17:36:32.371 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures/20260616-173632


2026-06-16 17:36:32.371 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/81 [00:00<?, ?it/s]

 12%|█▏        | 10/81 [00:00<00:00, 94.05it/s]

...

2026-06-16 17:37:26.153 | INFO     | mltrainer.trainer:report:209 - Epoch 56 train 0.0017 test 0.0232 metric ['0.9969']


2026-06-16 17:37:26.153 | INFO     | mltrainer.trainer:__call__:252 - best loss: 0.0173, current loss 0.0232.Counter 10/10.


2026-06-16 17:37:26.153 | INFO     | mltrainer.trainer:loop:103 - Interrupting loop due to early stopping patience.


2026-06-16 17:37:26.153 | INFO     | mltrainer.trainer:loop:108 - early_stopping_save was false, using latest model.Set to true to retrieve best model.


 56%|█████▌    | 56/100 [00:53<00:42,  1.04it/s]

### Experiment 3: LSTM with larger hidden size

In [23]:
config_lstm_large = ModelConfig(
    input_size=3,
    hidden_size=256,
    num_layers=3,
    output_size=20,
    dropout=0.3,
)
model_lstm_large = LSTMmodel(config=config_lstm_large)
run_experiment(model_lstm_large, "LSTM-h256-l3-d03", config_lstm_large.__dict__)

2026-06-16 17:37:26.168 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures/20260616-173726


2026-06-16 17:37:26.168 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/81 [00:00<?, ?it/s]

  5%|▍         | 4/81 [00:00<00:02, 34.80it/s]

...

2026-06-16 17:39:18.702 | INFO     | mltrainer.trainer:report:209 - Epoch 45 train 0.0026 test 0.0259 metric ['0.9938']


2026-06-16 17:39:18.702 | INFO     | mltrainer.trainer:__call__:252 - best loss: 0.0236, current loss 0.0259.Counter 10/10.


2026-06-16 17:39:18.703 | INFO     | mltrainer.trainer:loop:103 - Interrupting loop due to early stopping patience.


2026-06-16 17:39:18.703 | INFO     | mltrainer.trainer:loop:108 - early_stopping_save was false, using latest model.Set to true to retrieve best model.


 45%|████▌     | 45/100 [01:52<02:17,  2.50s/it]

### Experiment 4: Conv1D + GRU hybrid

In [24]:
config_conv_gru = Conv1DGRUConfig(
    input_size=3,
    hidden_size=128,
    num_layers=2,
    output_size=20,
    conv_filters=64,
    kernel_size=5,
    dropout=0.3,
)
model_conv_gru = Conv1DGRUmodel(config=config_conv_gru)
run_experiment(model_conv_gru, "Conv1D-GRU-f64-k5-h128-l2", config_conv_gru.__dict__)

2026-06-16 17:39:18.720 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures/20260616-173918


2026-06-16 17:39:18.720 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/81 [00:00<?, ?it/s]

 14%|█▎        | 11/81 [00:00<00:00, 100.81it/s]

...

2026-06-16 17:39:53.250 | INFO     | mltrainer.trainer:report:209 - Epoch 37 train 0.0019 test 0.0189 metric ['0.9953']


2026-06-16 17:39:53.250 | INFO     | mltrainer.trainer:__call__:252 - best loss: 0.0141, current loss 0.0189.Counter 10/10.


2026-06-16 17:39:53.250 | INFO     | mltrainer.trainer:loop:103 - Interrupting loop due to early stopping patience.


2026-06-16 17:39:53.251 | INFO     | mltrainer.trainer:loop:108 - early_stopping_save was false, using latest model.Set to true to retrieve best model.


 37%|███▋      | 37/100 [00:34<00:58,  1.07it/s]

### Experiment 5: Conv1D + GRU with larger conv filters and hidden size

In [25]:
config_conv_gru_large = Conv1DGRUConfig(
    input_size=3,
    hidden_size=256,
    num_layers=2,
    output_size=20,
    conv_filters=128,
    kernel_size=5,
    dropout=0.3,
)
model_conv_gru_large = Conv1DGRUmodel(config=config_conv_gru_large)
run_experiment(model_conv_gru_large, "Conv1D-GRU-f128-k5-h256-l2", config_conv_gru_large.__dict__)

2026-06-16 17:39:53.264 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures/20260616-173953


2026-06-16 17:39:53.264 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/81 [00:00<?, ?it/s]

  9%|▊         | 7/81 [00:00<00:01, 64.15it/s]

...

2026-06-16 17:40:59.050 | INFO     | mltrainer.trainer:report:209 - Epoch 45 train 0.0009 test 0.0041 metric ['0.9984']


2026-06-16 17:40:59.050 | INFO     | mltrainer.trainer:__call__:252 - best loss: 0.0021, current loss 0.0041.Counter 10/10.


2026-06-16 17:40:59.050 | INFO     | mltrainer.trainer:loop:103 - Interrupting loop due to early stopping patience.


2026-06-16 17:40:59.051 | INFO     | mltrainer.trainer:loop:108 - early_stopping_save was false, using latest model.Set to true to retrieve best model.


 45%|████▌     | 45/100 [01:05<01:20,  1.46s/it]

In [26]:
# Note: to continue training the last model, re-create the trainer
# and call trainer.loop() — it will resume from the last epoch.
# The experiments above already run 100 epochs with early stopping.

In [27]:
mlflow.end_run()

Excercises:

- try to improve the RNN model
- test different things. What works? What does not?
- experiment with either GRU or LSTM layers, create your own models. Have a look at `mltrainer.rnn_models` for inspiration. 
- experiment with adding Conv1D layers. Think about the necessary input-output dimensions of your tensors before and after each layer.

You should be able to get above 90% accuracy with the dataset.
Create a report of 1 a4 about your experiments.